# Práctica 01 — Del notebook a la producción

**Curso:** MLOps y Monitorización con Python · **Clase 01** · **Tiempo estimado:** 40-55 minutos
**Notebook de clase:** `notebooks_clase/clase01_del_notebook_a_produccion.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio parte de un fragmento de código con **el mismo tipo de problema** que viste en `datos/notebook_sucio.ipynb`. Tu trabajo es corregirlo, no entrenar un modelo real: los datos son pequeños arreglos de juguete.
- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas` y `scikit-learn` (solo `DummyClassifier` y `train_test_split`).

---
## Celda 0 — Setup

In [ ]:
import os
import inspect

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

print("Librerías cargadas correctamente.")

## Ejercicio 1 — Quitar la ruta absoluta hardcodeada

`ruta_datos_clientes` está copiada de un notebook real: devuelve siempre la misma ruta absoluta del portátil de quien lo escribió. Complétala para que:

- reciba `nombre` como parámetro (con `"clientes.csv"` por defecto),
- devuelva `os.path.join("datos", nombre)` en vez de una ruta absoluta fija.

In [ ]:
def ruta_datos_clientes():
    return "/Users/alguien/proyectos/churn/datos/clientes.csv"      # TODO: parámetro «nombre» + os.path.join("datos", nombre)

# Comprobación automática
firma = inspect.signature(ruta_datos_clientes)
assert "nombre" in firma.parameters, "Completa las líneas marcadas con TODO"
r = ruta_datos_clientes()
assert r == os.path.join("datos", "clientes.csv"), "Completa las líneas marcadas con TODO"
assert not r.startswith("/Users") and not r.startswith("/home") and ":\\" not in r, "la ruta no debe ser absoluta ni depender de un usuario concreto"
assert ruta_datos_clientes("otro.csv") == os.path.join("datos", "otro.csv"), "el parámetro «nombre» debe usarse de verdad"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Añadir `random_state` a una división reutilizable

`dividir` se llama desde varios sitios del proyecto, pero no fija ninguna semilla: cada llamada reparte las filas de forma distinta. Complétala para que reciba `semilla` y la pase a `train_test_split`.

In [ ]:
def dividir(X, y, semilla=42):
    return train_test_split(X, y, test_size=0.25)      # TODO: pasa random_state=semilla

# Comprobación automática
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
a1, a2, a3, a4 = dividir(X_juguete, y_juguete, semilla=7)
b1, b2, b3, b4 = dividir(X_juguete, y_juguete, semilla=7)
assert np.array_equal(a1, b1) and np.array_equal(a2, b2), "Completa las líneas marcadas con TODO"
c1, c2, c3, c4 = dividir(X_juguete, y_juguete, semilla=99)
assert not np.array_equal(a1, c1), "con semillas distintas el reparto también debería ser distinto"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Sacar el umbral de una variable global

`en_riesgo` lee `umbral_global`, una variable definida fuera de la función. Si alguien la sobrescribe más adelante (como pasaba con `umbral_riesgo` en el notebook sucio), el resultado de `en_riesgo` cambia sin que la función haya cambiado. Complétala para que reciba `umbral` como parámetro, con `0.5` por defecto, y deje de depender de la variable global.

In [ ]:
umbral_global = 0.5


def en_riesgo(prob):
    return prob > umbral_global      # TODO: parámetro «umbral=0.5», sin usar umbral_global

# Comprobación automática
firma = inspect.signature(en_riesgo)
assert "umbral" in firma.parameters, "Completa las líneas marcadas con TODO"
assert en_riesgo(0.6, umbral=0.5) is True and en_riesgo(0.4, umbral=0.5) is False, "Completa las líneas marcadas con TODO"
umbral_global = 0.9      # alguien sobrescribe la global más adelante, como en el notebook sucio
assert en_riesgo(0.6, umbral=0.5) is True, "la función ya no debe depender de umbral_global"
assert en_riesgo(0.6) is True, "con el valor por defecto (0.5) también debe dar True para 0.6"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — No tirar el modelo entrenado

`entrenar_y_tirar` entrena un modelo y no lo devuelve: como en la celda final del notebook sucio, el modelo queda solo en memoria dentro de la función y se pierde en cuanto termina. Complétala para que **devuelva el modelo entrenado**.

In [ ]:
def entrenar_y_tirar(X, y):
    modelo = DummyClassifier(strategy="most_frequent").fit(X, y)
    print("modelo entrenado")      # TODO: devuelve «modelo» en vez de perderlo

# Comprobación automática
X_juguete = np.arange(20).reshape(10, 2)
y_juguete = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0])
resultado = entrenar_y_tirar(X_juguete, y_juguete)
assert resultado is not None, "Completa las líneas marcadas con TODO"
assert hasattr(resultado, "predict"), "debe devolver el modelo, no None ni una métrica"
assert list(resultado.predict(X_juguete)) == [0] * 10, "con strategy=\"most_frequent\" siempre predice la clase mayoritaria (0)"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Validar los datos antes de usarlos

`cargar_validado` recibe un `DataFrame` y lo devuelve tal cual, sin comprobar nada — como el comentario "no hace falta comprobar nulos" del notebook sucio. Complétala para que levante `ValueError` si:

- falta alguna columna de `COLUMNAS_REQUERIDAS`, o
- hay algún valor nulo en `"antiguedad_meses"`.

Si todo está bien, debe devolver el `DataFrame` sin modificarlo.

In [ ]:
COLUMNAS_REQUERIDAS = ["antiguedad_meses", "uso_mensual_horas", "abandono"]


def cargar_validado(df):
    return df      # TODO: ValueError si faltan columnas o hay nulos en antiguedad_meses

# Comprobación automática
df_ok = pd.DataFrame({"antiguedad_meses": [1, 2, 3], "uso_mensual_horas": [10, 20, 30], "abandono": [0, 1, 0]})
assert cargar_validado(df_ok) is not None and len(cargar_validado(df_ok)) == 3, "Completa las líneas marcadas con TODO"
df_sin_columna = df_ok.drop(columns=["uso_mensual_horas"])
try:
    cargar_validado(df_sin_columna)
    raise AssertionError("debía levantar ValueError si falta una columna requerida")
except ValueError:
    pass
df_con_nulo = df_ok.copy()
df_con_nulo.loc[0, "antiguedad_meses"] = None
try:
    cargar_validado(df_con_nulo)
    raise AssertionError("debía levantar ValueError si hay nulos en antiguedad_meses")
except ValueError:
    pass
print("Ejercicio 5 correcto ✔")

---
## Reto final — un `pipeline_reproducible` sin ninguno de los problemas del notebook sucio

Integra lo aprendido. Completa `pipeline_reproducible(X, y, semilla)`, que debe:

- dividir con `dividir(X, y, semilla)` (ejercicio 2),
- entrenar un `DummyClassifier(strategy="most_frequent")` con el entrenamiento, **sin tirarlo** (ejercicio 4),
- devolver `(modelo, X_prueba, y_prueba)`.

Ninguna ruta absoluta, ninguna variable global, ninguna semilla ausente: todo lo necesario entra por parámetro.

In [ ]:
def pipeline_reproducible(X, y, semilla):
    X_ent, X_prueba, y_ent, y_prueba = None, None, None, None      # TODO: usa dividir(X, y, semilla)
    modelo = None      # TODO: entrena un DummyClassifier(strategy="most_frequent") con X_ent, y_ent
    return modelo, X_prueba, y_prueba

# Comprobación automática
X_juguete = np.arange(60).reshape(30, 2)
y_juguete = np.array(([0] * 20) + ([1] * 10))
resultado = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert resultado is not None, "Completa las líneas marcadas con TODO"
modelo, X_prueba, y_prueba = resultado
assert hasattr(modelo, "predict"), "devuelve el modelo entrenado, no None"
modelo_b, X_prueba_b, y_prueba_b = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert np.array_equal(X_prueba, X_prueba_b), "con la misma semilla la partición de prueba debe repetirse"
assert list(modelo.predict(X_prueba)) == list(modelo_b.predict(X_prueba_b)), "con la misma semilla las predicciones deben coincidir"
print("Reto final correcto ✔")

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def ruta_datos_clientes(nombre="clientes.csv"):
    return os.path.join("datos", nombre)

# Comprobación automática
firma = inspect.signature(ruta_datos_clientes)
assert "nombre" in firma.parameters, "Completa las líneas marcadas con TODO"
r = ruta_datos_clientes()
assert r == os.path.join("datos", "clientes.csv"), "Completa las líneas marcadas con TODO"
assert not r.startswith("/Users") and not r.startswith("/home") and ":\\" not in r, "la ruta no debe ser absoluta ni depender de un usuario concreto"
assert ruta_datos_clientes("otro.csv") == os.path.join("datos", "otro.csv"), "el parámetro «nombre» debe usarse de verdad"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def dividir(X, y, semilla=42):
    return train_test_split(X, y, test_size=0.25, random_state=semilla)

# Comprobación automática
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
a1, a2, a3, a4 = dividir(X_juguete, y_juguete, semilla=7)
b1, b2, b3, b4 = dividir(X_juguete, y_juguete, semilla=7)
assert np.array_equal(a1, b1) and np.array_equal(a2, b2), "Completa las líneas marcadas con TODO"
c1, c2, c3, c4 = dividir(X_juguete, y_juguete, semilla=99)
assert not np.array_equal(a1, c1), "con semillas distintas el reparto también debería ser distinto"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
umbral_global = 0.5


def en_riesgo(prob, umbral=0.5):
    return prob > umbral

# Comprobación automática
firma = inspect.signature(en_riesgo)
assert "umbral" in firma.parameters, "Completa las líneas marcadas con TODO"
assert en_riesgo(0.6, umbral=0.5) is True and en_riesgo(0.4, umbral=0.5) is False, "Completa las líneas marcadas con TODO"
umbral_global = 0.9      # alguien sobrescribe la global más adelante, como en el notebook sucio
assert en_riesgo(0.6, umbral=0.5) is True, "la función ya no debe depender de umbral_global"
assert en_riesgo(0.6) is True, "con el valor por defecto (0.5) también debe dar True para 0.6"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def entrenar_y_tirar(X, y):
    modelo = DummyClassifier(strategy="most_frequent").fit(X, y)
    print("modelo entrenado")
    return modelo

# Comprobación automática
X_juguete = np.arange(20).reshape(10, 2)
y_juguete = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0])
resultado = entrenar_y_tirar(X_juguete, y_juguete)
assert resultado is not None, "Completa las líneas marcadas con TODO"
assert hasattr(resultado, "predict"), "debe devolver el modelo, no None ni una métrica"
assert list(resultado.predict(X_juguete)) == [0] * 10, "con strategy=\"most_frequent\" siempre predice la clase mayoritaria (0)"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
COLUMNAS_REQUERIDAS = ["antiguedad_meses", "uso_mensual_horas", "abandono"]


def cargar_validado(df):
    faltantes = [c for c in COLUMNAS_REQUERIDAS if c not in df.columns]
    if faltantes:
        raise ValueError("faltan columnas: " + str(faltantes))
    if df["antiguedad_meses"].isnull().any():
        raise ValueError("hay valores nulos en antiguedad_meses")
    return df

# Comprobación automática
df_ok = pd.DataFrame({"antiguedad_meses": [1, 2, 3], "uso_mensual_horas": [10, 20, 30], "abandono": [0, 1, 0]})
assert cargar_validado(df_ok) is not None and len(cargar_validado(df_ok)) == 3, "Completa las líneas marcadas con TODO"
df_sin_columna = df_ok.drop(columns=["uso_mensual_horas"])
try:
    cargar_validado(df_sin_columna)
    raise AssertionError("debía levantar ValueError si falta una columna requerida")
except ValueError:
    pass
df_con_nulo = df_ok.copy()
df_con_nulo.loc[0, "antiguedad_meses"] = None
try:
    cargar_validado(df_con_nulo)
    raise AssertionError("debía levantar ValueError si hay nulos en antiguedad_meses")
except ValueError:
    pass
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def pipeline_reproducible(X, y, semilla):
    X_ent, X_prueba, y_ent, y_prueba = dividir(X, y, semilla)
    modelo = entrenar_y_tirar(X_ent, y_ent)
    return modelo, X_prueba, y_prueba

# Comprobación automática
X_juguete = np.arange(60).reshape(30, 2)
y_juguete = np.array(([0] * 20) + ([1] * 10))
resultado = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert resultado is not None, "Completa las líneas marcadas con TODO"
modelo, X_prueba, y_prueba = resultado
assert hasattr(modelo, "predict"), "devuelve el modelo entrenado, no None"
modelo_b, X_prueba_b, y_prueba_b = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert np.array_equal(X_prueba, X_prueba_b), "con la misma semilla la partición de prueba debe repetirse"
assert list(modelo.predict(X_prueba)) == list(modelo_b.predict(X_prueba_b)), "con la misma semilla las predicciones deben coincidir"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (deuda técnica específica del ML, niveles de madurez, orden de ejecución de celdas, variables globales, validación de datos)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*